# Robust AI-Image Detector — Free Kaggle Training

This notebook clones the public repository, streams a 6,000-image balanced SID_Set subset (not the full 140 GB dataset), trains clean and robustness-aware models, evaluates both across the complete challenge transform grid, validates the final checkpoint, and packages only reproducibility artifacts. Enable a Kaggle GPU and internet access before running all cells.

In [ ]:
from pathlib import Path
import json
import math
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/LINGSIHAN/TikTok-Hackathon-Track-5.git"
BRANCH = "master"
PROJECT_DIR = Path("/kaggle/working/TikTok-Hackathon-Track-5")
SUBSET_SIZE = 6_000
SEED = 42

def run(*args):
    print("+", " ".join(map(str, args)))
    subprocess.run([str(arg) for arg in args], check=True)

In [ ]:
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
run("git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, PROJECT_DIR)
os.chdir(PROJECT_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Training repository commit:", COMMIT)
run(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt")

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Kaggle choose Settings > Accelerator > GPU, then restart.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
run(sys.executable, "-m", "pytest", "-q")

In [ ]:
run(
    sys.executable,
    "scripts/prepare_sid_subset.py",
    "--total", str(SUBSET_SIZE),
    "--seed", str(SEED),
)

import pandas as pd

manifest = pd.read_csv("data/processed/manifest.csv")
counts = manifest.groupby(["split", "label"]).size().unstack(fill_value=0)
print(counts)
assert len(manifest) == SUBSET_SIZE
assert set(manifest["label"]) == {0, 1}
assert set(manifest["split"]) == {"train", "val", "test"}
assert (counts > 0).all().all()
assert not manifest["sha256"].duplicated().any()
assert manifest.groupby("source_id")["split"].nunique().max() == 1
assert Path("data/processed/manifest_summary.json").is_file()

In [ ]:
run(sys.executable, "-m", "src.training.train", "--config", "configs/train_clean.yaml", "--device", "cuda")

In [ ]:
run(
    sys.executable, "-m", "src.evaluation.evaluate",
    "--manifest", "data/processed/manifest.csv",
    "--checkpoint", "artifacts/runs/clean/model.safetensors",
    "--split", "test",
    "--output-dir", "artifacts/metrics/clean_baseline",
    "--device", "cuda",
)

In [ ]:
run(sys.executable, "-m", "src.training.train", "--config", "configs/train_robust.yaml", "--device", "cuda")

In [ ]:
run(
    sys.executable, "-m", "src.evaluation.evaluate",
    "--manifest", "data/processed/manifest.csv",
    "--checkpoint", "artifacts/checkpoints/model.safetensors",
    "--split", "test",
    "--output-dir", "artifacts/metrics",
    "--device", "cuda",
)

In [ ]:
required = [
    Path("artifacts/checkpoints/model.safetensors"),
    Path("artifacts/checkpoints/model_metadata.json"),
    Path("artifacts/metrics/training_history.json"),
    Path("artifacts/metrics/metrics.json"),
    Path("artifacts/metrics/predictions.csv"),
    Path("artifacts/metrics/robustness.png"),
    Path("artifacts/runs/clean/model.safetensors"),
    Path("artifacts/runs/clean/model_metadata.json"),
    Path("artifacts/runs/clean/history.json"),
    Path("artifacts/metrics/clean_baseline/metrics.json"),
    Path("artifacts/metrics/clean_baseline/predictions.csv"),
    Path("artifacts/metrics/clean_baseline/robustness.png"),
    Path("data/processed/manifest.csv"),
    Path("data/processed/manifest_summary.json"),
]
missing = [str(path) for path in required if not path.is_file() or path.stat().st_size == 0]
if missing:
    raise RuntimeError("Missing or empty required artifacts: " + ", ".join(missing))

for metrics_path in [Path("artifacts/metrics/metrics.json"), Path("artifacts/metrics/clean_baseline/metrics.json")]:
    payload = json.loads(metrics_path.read_text())
    assert payload["scenarios"], f"No scenarios in {metrics_path}"
    for scenario in payload["scenarios"]:
        for name, value in scenario["metrics"].items():
            assert isinstance(value, (int, float)) and math.isfinite(value), f"Non-finite {name} in {metrics_path}"

from PIL import Image
from src.inference.predictor import Predictor

test_path = Path(manifest.loc[manifest["split"] == "test", "path"].iloc[0])
predictor = Predictor.from_checkpoint("artifacts/checkpoints/model.safetensors", device="cuda")
with Image.open(test_path) as image:
    smoke_probability = predictor.predict_pil(image.convert("RGB"))
assert 0.0 <= smoke_probability <= 1.0
print("Checkpoint smoke probability:", smoke_probability)

In [ ]:
export_dir = Path("/kaggle/working/export")
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True)

for source, destination in [
    (Path("artifacts/checkpoints"), export_dir / "artifacts/checkpoints"),
    (Path("artifacts/runs/clean"), export_dir / "artifacts/runs/clean"),
    (Path("artifacts/metrics"), export_dir / "artifacts/metrics"),
    (Path("data/processed/manifest.csv"), export_dir / "data/processed/manifest.csv"),
    (Path("data/processed/manifest_summary.json"), export_dir / "data/processed/manifest_summary.json"),
    (Path("configs/train_clean.yaml"), export_dir / "configs/train_clean.yaml"),
    (Path("configs/train_robust.yaml"), export_dir / "configs/train_robust.yaml"),
    (Path("requirements.txt"), export_dir / "requirements.txt"),
    (Path("requirements-train.txt"), export_dir / "requirements-train.txt"),
]:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        shutil.copy2(source, destination)

run_context = {
    "repository": REPO_URL,
    "branch": BRANCH,
    "commit": COMMIT,
    "subset_size": SUBSET_SIZE,
    "seed": SEED,
    "pytorch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
}
run_context_path = export_dir / "artifacts/metrics/run_context.json"
run_context_path.parent.mkdir(parents=True, exist_ok=True)
run_context_path.write_text(json.dumps(run_context, indent=2) + "\n", encoding="utf-8")

archive = shutil.make_archive("/kaggle/working/hackathon_export", "zip", export_dir)
print("Validated export ready in Kaggle Output:", archive)